# Decision Tree Classifiers — Palmer Penguins

Training a multi-class decision tree classifier to predict penguin species from physical measurements.

**Pipeline:**
1. Load and explore the Palmer Penguins dataset (344 observations, 7 features)
2. Handle missing data — sorted null table, threshold check, dropna
3. Clean categorical features (remove ambiguous `'.'` sex entries)
4. Visualize class separability — scatterplot, pairplot, categorical boxplot
5. Feature engineering — one-hot encoding with L−1 dummy variables
6. Train `DecisionTreeClassifier`, evaluate with confusion matrix
7. Explore hyperparameters: `max_depth`, `max_leaf_nodes`, `criterion`

**Dataset:** Palmer Penguins — 3 species (Adélie, Chinstrap, Gentoo), 3 islands, 333 usable observations after cleaning.  
**Source:** Gorman KB, Williams TD, Fraser WR (2014). PLoS ONE 9(3): e90081.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

---
## 1. Load and Explore

In [ ]:
df = pd.read_csv("data/penguins_size.csv")

print(f"Shape: {df.shape}")
print(f"Features ({df.shape[1]}): {list(df.columns)}")
df.tail(6)

---
## 2. Missing Data Analysis

Generating a sorted missing-value table, computing the worst-case percentage, and dropping rows if below 10%.

In [ ]:
df.info()

In [ ]:
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_table  = pd.DataFrame({'feature': missing_counts.index, 'missing_count': missing_counts.values})
missing_table

In [ ]:
top_feature = missing_table.loc[0, 'feature']
top_missing = missing_table.loc[0, 'missing_count']
pct_missing = (top_missing / len(df)) * 100

print(f"Worst feature: '{top_feature}' — {top_missing} missing ({pct_missing:.2f}%)")
print(f"2.91% << 10% threshold → dropping all rows with any missing value")

In [ ]:
df_clean = df.dropna().copy()
print(f"Before: {len(df)} rows → After: {len(df_clean)} rows ({len(df) - len(df_clean)} dropped)")
assert df_clean.isna().sum().sum() == 0, "Nulls remain!"

---
## 3. Categorical Feature Inspection and Cleaning

The `sex` column contains a `'.'` entry — a researcher annotation meaning sex could not be determined. These rows are removed because they represent ambiguous data, not a meaningful category.

In [ ]:
print("Unique islands:", df_clean['island'].unique())
print("Unique sex values:", df_clean['sex'].unique())

# Remove ambiguous '.' entries
unknown_sex = ['UNKNOWN', 'Unknown', 'unknown', '.', 'NA', 'N/A', '']
df_clean = df_clean[~df_clean['sex'].isin(unknown_sex)].copy()

print(f"\nAfter removing ambiguous sex entries: {len(df_clean)} rows")
print("Sex values remaining:", df_clean['sex'].unique())
print("Species (label) classes:", df_clean['species'].unique())

---
## 4. Visualization

### 4.1 Culmen Length vs Culmen Depth

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_clean, x='culmen_length_mm', y='culmen_depth_mm',
                hue='species', palette='Dark2')
plt.title('Culmen Length vs Culmen Depth (by Species)')
plt.tight_layout()
plt.show()

print("""
Inference:
Gentoo is well separated — lower culmen depth across all culmen lengths.
Adélie and Chinstrap overlap significantly in this 2D view, but Chinstrap
tends toward longer culmen lengths. A decision tree will need additional
features (flipper length, body mass) to fully separate all three species.
""")

### 4.2 Pairplot — all numeric features

In [ ]:
sns.pairplot(
    df_clean, hue='species', palette='Dark2',
    vars=['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g']
)
plt.show()

### 4.3 Species vs Culmen Length — by sex

In [ ]:
g = sns.catplot(
    data=df_clean, x='species', y='culmen_length_mm',
    col='sex', kind='box', hue='species',
    palette='Dark2', legend=False, height=4, aspect=1
)
g.set_axis_labels('Species', 'Culmen Length (mm)')
g.set_titles('sex = {col_name}')
plt.show()

print("""
Which groups are better separated?
For both sexes, Adélie is clearly separated from Chinstrap and Gentoo
in culmen length (lower median, tighter IQR). Chinstrap and Gentoo overlap
more in culmen length, though their medians differ. The sex-split plot shows
the same pattern holds for both males and females.
""")

---
## 5. Feature Engineering — One-Hot Encoding

Decision trees require numeric inputs. `island` and `sex` are categorical and need encoding.

**Why L−1 dummy variables?** For a feature with L categories, L−1 binary columns suffice. If all L−1 are 0, the observation belongs to the dropped category — no information is lost. Keeping all L introduces perfect multicollinearity (dummy variable trap).

In [ ]:
# Encode only features (not the label 'species')
X = pd.get_dummies(df_clean.drop(columns=['species']), columns=['island', 'sex'])
y = df_clean['species'].copy()

# Drop one dummy per categorical feature (L-1 rule)
X = X.drop(columns=['island_Biscoe', 'sex_FEMALE']).copy()

print(f"Feature matrix X: {X.shape}  →  columns: {list(X.columns)}")
print(f"Label vector y   : {y.shape}  →  classes: {y.unique()}")
X.head()

---
## 6. Decision Tree Classifier

### 6.1 Train/test split and baseline model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=101
)
print(f"Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples")

In [ ]:
tree_clf = DecisionTreeClassifier(random_state=101)
tree_clf.fit(X_train, y_train)
y_pred = tree_clf.predict(X_test)

print(classification_report(y_test, y_pred, digits=3))

### 6.2 Confusion matrix

A confusion matrix is preferred over single-number accuracy when class balance matters. Each cell (i, j) shows how many samples of true class i were predicted as class j — the diagonal is correct predictions.

In [ ]:
cm   = confusion_matrix(y_test, y_pred, labels=tree_clf.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=tree_clf.classes_)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Default Decision Tree')
plt.show()

print(f"Raw matrix:\n{cm}")
print()
print("""
Most errors are Adélie ↔ Chinstrap confusions — consistent with the EDA
showing those two species overlap most in feature space.
Gentoo is classified perfectly (0 errors), matching its clear visual separation.
""")

### 6.3 Visualise the trained tree

In [ ]:
plt.figure(figsize=(18, 10))
plot_tree(
    tree_clf,
    feature_names=X.columns,
    class_names=tree_clf.classes_,
    filled=True,
    fontsize=7
)
plt.title('Default Decision Tree (unconstrained depth)')
plt.tight_layout()
plt.show()

---
## 7. Hyperparameter Exploration

The default tree is unconstrained — it grows until all leaves are pure, which risks overfitting. Three hyperparameters control tree complexity:
- `max_depth` — limits the number of decision levels
- `max_leaf_nodes` — caps total leaf count
- `criterion` — switches split quality metric from Gini to Entropy

### 7.1 Maximum depth = 2

In [ ]:
tree_depth2 = DecisionTreeClassifier(random_state=101, max_depth=2)
tree_depth2.fit(X_train, y_train)

print(f"Leaves: {tree_depth2.get_n_leaves()}  |  Test accuracy: {tree_depth2.score(X_test, y_test):.3f}")

plt.figure(figsize=(14, 6))
plot_tree(tree_depth2, feature_names=X.columns,
          class_names=tree_depth2.classes_, filled=True)
plt.title('Decision Tree (max_depth = 2)')
plt.tight_layout()
plt.show()

### 7.2 Maximum leaf nodes = 3

In [ ]:
tree_leaf3 = DecisionTreeClassifier(random_state=101, max_leaf_nodes=3)
tree_leaf3.fit(X_train, y_train)

print(f"Leaves: {tree_leaf3.get_n_leaves()}  |  Test accuracy: {tree_leaf3.score(X_test, y_test):.3f}")

plt.figure(figsize=(14, 6))
plot_tree(tree_leaf3, feature_names=X.columns,
          class_names=tree_leaf3.classes_, filled=True)
plt.title('Decision Tree (max_leaf_nodes = 3)')
plt.tight_layout()
plt.show()

### 7.3 Entropy criterion instead of Gini

By default, scikit-learn uses Gini impurity to select splits. Switching to `criterion='entropy'` uses information gain instead — both measure node impurity but entropy is more sensitive to class probability differences, which can produce different (sometimes deeper) split structures.

In [ ]:
tree_entropy = DecisionTreeClassifier(random_state=101, criterion='entropy')
tree_entropy.fit(X_train, y_train)

print(f"Leaves: {tree_entropy.get_n_leaves()}  |  Test accuracy: {tree_entropy.score(X_test, y_test):.3f}")

plt.figure(figsize=(18, 10))
plot_tree(tree_entropy, feature_names=X.columns,
          class_names=tree_entropy.classes_, filled=True, fontsize=7)
plt.title("Decision Tree (criterion = 'entropy')")
plt.tight_layout()
plt.show()

---
## Summary

| Model | Leaves | Test Accuracy |
|---|---|---|
| Default (Gini, unconstrained) | varies | ~0.95 |
| max_depth = 2 | 4 | lower |
| max_leaf_nodes = 3 | 3 | lower |
| Entropy criterion | varies | ~0.95 |

Key takeaways:
- Flipper length is the most powerful first split — it cleanly isolates Gentoo
- Adélie vs Chinstrap requires culmen length as a secondary split
- Constraining depth/leaves reduces overfitting risk at the cost of accuracy
- Gini and Entropy produce similar accuracy on this dataset — the root split feature is identical (flipper_length_mm)